# Bounded local hyperparameter search

This standalone lesson requires the matching installed DRYML version with its `sklearn` extra (`dryml[sklearn]`). It runs offline with tiny fixed NumPy arrays and uses only a temporary Store. The workflow is deliberately finite and local: define a search space, preflight its bounded grid, train serially, evaluate finite metrics, and publish one best candidate only after every candidate succeeds.

In [ ]:
import json
import random
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
from sklearn.linear_model import Ridge

from dryml.core2 import Definition, Repo, SearchSpace, UniformFromSet
from dryml.core2.store import DirStore
from dryml.data import ArrayDataset
from dryml.metrics import mean_squared_error
from dryml.models import Experiment
from dryml.models.sklearn import BasicTraining, RegressionModel

## A small nested search definition

`UniformFromSet` supplies both matching support and a finite generator. Here two model constructor parameters each have two choices, so the nested experiment space has exactly four candidates. All other experiment, training, and dataset structure is fixed public constructor data.

In [ ]:
x_train = np.array(
    [[0.0], [1.0], [2.0], [3.0], [4.0]],
    dtype=np.float32,
)
y_train = np.array([0.0, 1.0, 1.5, 3.0, 4.5], dtype=np.float32)

dataset_definition = Definition(ArrayDataset, (x_train, y_train))
model_definition = Definition(
    RegressionModel,
    Ridge,
    alpha=UniformFromSet([0.0, 0.5], name='alpha'),
    fit_intercept=UniformFromSet([False, True], name='fit_intercept'),
)
experiment_definition = Definition(
    Experiment,
    model_definition,
    Definition(BasicTraining),
    train_data=dataset_definition,
)
search_space = experiment_definition.as_space()
assert isinstance(search_space, SearchSpace)

## Seeded sampling

Pass an explicit standard-library random generator to `sample()`. Recreating it from the same fixed seed reproduces the same complete definition without relying on process-global random state.

In [ ]:
FIXED_SEED = 20260718
sample_one = search_space.sample(random.Random(FIXED_SEED)).concretize()
sample_two = search_space.sample(random.Random(FIXED_SEED)).concretize()
assert sample_one == sample_two
assert sample_one.stable_hash() == sample_two.stable_hash()

## Bound the grid before execution

The preflight reads at most `cap + 1` generated combinations and never inspects private `SearchSpace` state. Empty and oversized grids fail before concretization, training, or publication. The cap bounds candidate execution and publication, not a generator's internal grid materialization; arbitrary-range preflight is outside this lesson.

In [ ]:
def stable_cdef_key(cdef):
    return (cdef.stable_hash(), repr(cdef))


def bounded_grid(space, cap):
    if not isinstance(cap, int) or isinstance(cap, bool) or cap < 1:
        raise ValueError('candidate cap must be a positive integer')

    grid_iterator = iter(space.grid())
    candidate_definitions = []
    for _ in range(cap + 1):
        try:
            candidate_definitions.append(next(grid_iterator))
        except StopIteration:
            break

    if not candidate_definitions:
        raise ValueError('candidate grid is empty')
    if len(candidate_definitions) > cap:
        raise ValueError(
            f'candidate grid exceeds execution cap {cap}; '
            f'observed {len(candidate_definitions)} candidates'
        )

    candidate_cdefs = tuple(
        definition.concretize() for definition in candidate_definitions
    )
    return tuple(sorted(candidate_cdefs, key=stable_cdef_key))

The empty and over-cap cases are handled before a Repo or candidate object exists. The oversized four-candidate grid is rejected with `cap=3` after observing exactly the permitted `cap + 1` combinations.

In [ ]:
empty_model_definition = model_definition.with_kwarg(
    'alpha', UniformFromSet([], name='empty_alpha')
).with_kwarg('fit_intercept', True)
empty_space = experiment_definition.with_arg(
    0, empty_model_definition
).as_space()
empty_best = None
empty_training_attempts = 0
try:
    bounded_grid(empty_space, cap=4)
except ValueError as error:
    assert str(error) == 'candidate grid is empty'
else:
    raise AssertionError('an empty grid should fail preflight')
assert empty_best is None
assert empty_training_attempts == 0

try:
    bounded_grid(search_space, cap=3)
except ValueError as error:
    assert 'execution cap 3' in str(error)
    assert 'observed 4 candidates' in str(error)
else:
    raise AssertionError('the four-candidate grid should exceed cap=3')

## Stable support and candidate order

With the intended cap, preflight returns all four CDefs in a deterministic identity order. `support_selector()` describes every generated CDef and rejects an otherwise identical experiment whose `alpha` is outside the generated set.

In [ ]:
CANDIDATE_CAP = 4
ordered_candidates = bounded_grid(search_space, cap=CANDIDATE_CAP)
assert len(ordered_candidates) == 4
assert len(set(ordered_candidates)) == 4
assert ordered_candidates == tuple(
    sorted(ordered_candidates, key=stable_cdef_key)
)

support = search_space.support_selector()
assert all(support.matches(cdef) for cdef in ordered_candidates)
out_of_support_model = model_definition.with_kwarg(
    'alpha', 2.0
).with_kwarg('fit_intercept', True)
out_of_support_cdef = experiment_definition.with_arg(
    0, out_of_support_model
).concretize()
assert not support.matches(out_of_support_cdef)

## Serial training and atomic result publication

The runner requires stable CDef order, constructs and trains each candidate exactly once, and rejects a non-finite scalar metric. It returns results only after the complete loop succeeds. Selection and Repo publication therefore happen outside the runner: any construction, training, or metric failure prevents a partial best candidate from being published. Lower mean squared error is better; equal metrics use the same stable CDef identity key as the execution order.

In [ ]:
def require_finite_metric(value):
    metric = float(value)
    if not np.isfinite(metric):
        raise ValueError('candidate metric must be finite')
    return metric


def train_candidates(candidate_cdefs, repo):
    candidate_cdefs = tuple(candidate_cdefs)
    if candidate_cdefs != tuple(sorted(candidate_cdefs, key=stable_cdef_key)):
        raise ValueError('candidates must be in stable CDef order')

    execution_counts = {}
    results = []
    for cdef in candidate_cdefs:
        identity = cdef.stable_hash()
        execution_counts[identity] = execution_counts.get(identity, 0) + 1
        if execution_counts[identity] != 1:
            raise RuntimeError('a candidate cannot execute more than once')

        experiment = repo.load_or_build(cdef)
        experiment.train()
        metric = require_finite_metric(mean_squared_error(
            experiment.model,
            experiment.train_data,
            batch_size=len(x_train),
        ))
        results.append({
            'cdef': cdef,
            'experiment': experiment,
            'metric': metric,
        })

    return tuple(results), execution_counts


def select_best(results):
    if not results:
        raise ValueError('cannot select from empty results')
    return min(
        results,
        key=lambda result: (
            result['metric'], stable_cdef_key(result['cdef'])
        ),
    )


try:
    require_finite_metric(float('nan'))
except ValueError as error:
    assert str(error) == 'candidate metric must be finite'
else:
    raise AssertionError('a non-finite metric should be rejected')

tie_results = (
    {'cdef': ordered_candidates[-1], 'metric': 1.0},
    {'cdef': ordered_candidates[0], 'metric': 1.0},
)
assert select_best(tie_results)['cdef'] == ordered_candidates[0]

A real invalid sklearn candidate demonstrates the error boundary. The best value remains unpublished because candidate execution does not return a complete result set.

In [ ]:
failing_model = model_definition.with_kwarg(
    'alpha', -1.0
).with_kwarg('fit_intercept', True)
failing_cdef = experiment_definition.with_arg(0, failing_model).concretize()
failing_candidates = tuple(sorted(
    (ordered_candidates[0], failing_cdef), key=stable_cdef_key
))
failed_best = None
failure_repo = Repo()
try:
    failed_results, _ = train_candidates(failing_candidates, failure_repo)
    failed_best = select_best(failed_results)
except ValueError as error:
    assert str(error)
else:
    raise AssertionError('the invalid candidate should fail training')
finally:
    failure_repo.close(flush=False)
assert failed_best is None

## Execute and publish the successful search

All four candidates now train serially in the preflight order. Only after all metrics are finite and every execution count is one does the lesson select and save the best experiment under an alias. Reopening the temporary Store verifies that publication preserved the selected CDef and trained model state. This is a same-version demonstration, not a Store interchange guarantee.

In [ ]:
with TemporaryDirectory() as temporary_root:
    store_path = Path(temporary_root) / 'local-search-store'
    search_repo = Repo(stores=DirStore(store_path, query_index='memory'))
    try:
        search_results, execution_counts = train_candidates(
            ordered_candidates, search_repo
        )
        assert len(search_results) == CANDIDATE_CAP
        assert tuple(
            result['cdef'] for result in search_results
        ) == ordered_candidates
        assert all(count == 1 for count in execution_counts.values())
        assert all(np.isfinite(result['metric']) for result in search_results)

        best_candidate = select_best(search_results)
        published_cdef = best_candidate['cdef']
        search_repo.save_object(
            best_candidate['experiment'], alias='best-local-candidate'
        )
    finally:
        search_repo.close(flush=True)

    reopened_repo = Repo(stores=DirStore(store_path, query_index='memory'))
    try:
        restored_best = reopened_repo.load_alias(
            'best-local-candidate', instance='new', cache='none'
        )
        assert restored_best.definition == published_cdef
        restored_metric = require_finite_metric(mean_squared_error(
            restored_best.model,
            restored_best.train_data,
            batch_size=len(x_train),
        ))
        np.testing.assert_allclose(
            restored_metric, best_candidate['metric'], atol=1e-12
        )
    finally:
        reopened_repo.close(flush=False)

summary = {
    'best_identity': published_cdef.stable_hash(),
    'candidate_count': len(search_results),
    'execution_order': [
        result['cdef'].stable_hash() for result in search_results
    ],
    'metrics': [round(result['metric'], 12) for result in search_results],
    'sample_identity': sample_one.stable_hash(),
}
print('LOCAL_SEARCH_SUMMARY=' + json.dumps(summary, sort_keys=True))

This lesson intentionally stops at supported local composition: finite `SearchSpace` generation, maintained sklearn experiment training, explicit scalar metrics, deterministic selection, and optional temporary Repo publication.